# 📊 UAS Data Science — Week 4 Progress
## Hyperparameter Tuning & Model Tambahan (XGBoost)

**Dataset:** Sales & Marketing Customer Dataset  
**Tujuan Week 4:** Meningkatkan performa model melalui hyperparameter tuning dan menambahkan model XGBoost, serta optimasi threshold klasifikasi

---

## 1. Recap Week 3

Pada Week 3 telah dilatih 3 model baseline dengan hasil sebagai berikut:

| Model | Catatan |
|-------|---------|
| Logistic Regression | Baseline linear, performa moderat |
| Decision Tree | Cepat tapi rawan overfitting |
| Random Forest | Performa terbaik dari 3 model baseline |

**Rencana Week 4:**
- Hyperparameter tuning Random Forest dengan `GridSearchCV`
- Menambahkan model **XGBoost**
- Optimasi threshold klasifikasi untuk memaksimalkan **Recall** kelas Churn (karena false negative lebih merugikan bisnis)

---

## 2. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
    classification_report
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='Set2')

import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport (termasuk XGBoost)')

In [ ]:
# Load data hasil preprocessing Week 2
df = pd.read_csv('data_modeling.csv')

X = df.drop(columns=['churn'])
y = df['churn']

# Split sama seperti Week 3 (random_state=42 agar konsisten)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE pada training set
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Training set (SMOTE): {X_train_sm.shape}')
print(f'Testing set          : {X_test.shape}')

## 3. Hyperparameter Tuning — Random Forest

### 3.1 Definisi Grid Parameter

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [8, 10, 15],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

print('Total kombinasi parameter:', 
      len(param_grid_rf['n_estimators']) * len(param_grid_rf['max_depth']) * 
      len(param_grid_rf['min_samples_split']) * len(param_grid_rf['min_samples_leaf']))
print()
for k, v in param_grid_rf.items():
    print(f'  {k}: {v}')

### 3.2 Menjalankan GridSearchCV

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    verbose=0
)

grid_rf.fit(X_train_sm, y_train_sm)

print('✅ GridSearchCV selesai')
print(f'\nBest parameters: {grid_rf.best_params_}')
print(f'Best CV F1-Score: {grid_rf.best_score_:.4f}')

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

print('=== Random Forest (Tuned) — Performa di Test Set ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob_rf):.4f}')

## 4. Model Tambahan — XGBoost

### 4.1 Training XGBoost Default

In [ ]:
xgb_default = XGBClassifier(
    random_state=42, eval_metric='logloss', use_label_encoder=False
)
xgb_default.fit(X_train_sm, y_train_sm)

y_pred_xgb_def = xgb_default.predict(X_test)
y_prob_xgb_def = xgb_default.predict_proba(X_test)[:, 1]

print('=== XGBoost (Default Params) — Performa di Test Set ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_xgb_def):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_xgb_def):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_xgb_def):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_xgb_def):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob_xgb_def):.4f}')

### 4.2 Tuning XGBoost dengan GridSearchCV

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
    param_grid=param_grid_xgb,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    verbose=0
)

grid_xgb.fit(X_train_sm, y_train_sm)

print('✅ GridSearchCV untuk XGBoost selesai')
print(f'\nBest parameters: {grid_xgb.best_params_}')
print(f'Best CV F1-Score: {grid_xgb.best_score_:.4f}')

In [ ]:
best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

print('=== XGBoost (Tuned) — Performa di Test Set ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_xgb):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_xgb):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_xgb):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob_xgb):.4f}')

## 5. Perbandingan Semua Model (Sebelum vs Setelah Tuning)

In [ ]:
comparison = pd.DataFrame({
    'Random Forest (Tuned)': {
        'Accuracy': accuracy_score(y_test, y_pred_rf),
        'Precision': precision_score(y_test, y_pred_rf),
        'Recall': recall_score(y_test, y_pred_rf),
        'F1-Score': f1_score(y_test, y_pred_rf),
        'ROC-AUC': roc_auc_score(y_test, y_prob_rf)
    },
    'XGBoost (Default)': {
        'Accuracy': accuracy_score(y_test, y_pred_xgb_def),
        'Precision': precision_score(y_test, y_pred_xgb_def),
        'Recall': recall_score(y_test, y_pred_xgb_def),
        'F1-Score': f1_score(y_test, y_pred_xgb_def),
        'ROC-AUC': roc_auc_score(y_test, y_prob_xgb_def)
    },
    'XGBoost (Tuned)': {
        'Accuracy': accuracy_score(y_test, y_pred_xgb),
        'Precision': precision_score(y_test, y_pred_xgb),
        'Recall': recall_score(y_test, y_pred_xgb),
        'F1-Score': f1_score(y_test, y_pred_xgb),
        'ROC-AUC': roc_auc_score(y_test, y_prob_xgb)
    }
}).T

print(comparison.to_string())

best_model_name = comparison['F1-Score'].idxmax()
print(f'\n🏆 Model terbaik (F1-Score): {best_model_name}')

In [ ]:
# Visualisasi perbandingan
fig, ax = plt.subplots(figsize=(13, 6))
comparison.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', 5), edgecolor='white', width=0.75)
ax.set_title('Perbandingan Performa Model: Sebelum vs Setelah Tuning', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=9)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 6. ROC Curve — Model Setelah Tuning

In [ ]:
plt.figure(figsize=(9, 7))

for name, y_prob, color in [
    ('Random Forest (Tuned)', y_prob_rf, '#3498db'),
    ('XGBoost (Default)', y_prob_xgb_def, '#e67e22'),
    ('XGBoost (Tuned)', y_prob_xgb, '#2ecc71')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC={auc_val:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Model Setelah Tuning', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

## 7. Optimasi Threshold Klasifikasi

Secara default, threshold klasifikasi adalah 0.5. Namun karena **false negative** (pelanggan churn yang tidak terdeteksi) lebih merugikan bisnis dibanding false positive, kita akan mencari threshold optimal yang memaksimalkan Recall tanpa terlalu mengorbankan Precision.

In [ ]:
# Gunakan model terbaik (berdasarkan F1) untuk analisis threshold
best_y_prob = y_prob_xgb if best_model_name == 'XGBoost (Tuned)' else y_prob_rf

precisions, recalls, thresholds = precision_recall_curve(y_test, best_y_prob)

# Hitung F1 di setiap threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else 0.5

print(f'Threshold default          : 0.5')
print(f'Threshold optimal (max F1) : {best_threshold:.4f}')
print(f'F1-Score pada threshold optimal: {f1_scores[best_threshold_idx]:.4f}')

In [ ]:
# Visualisasi Precision-Recall trade-off
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions[:-1], label='Precision', color='#3498db', linewidth=2)
plt.plot(thresholds, recalls[:-1], label='Recall', color='#e74c3c', linewidth=2)
plt.plot(thresholds, f1_scores[:-1], label='F1-Score', color='#2ecc71', linewidth=2, linestyle='--')
plt.axvline(best_threshold, color='black', linestyle=':', linewidth=1.5,
            label=f'Threshold Optimal = {best_threshold:.3f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall-F1 vs Threshold', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi performa dengan threshold optimal
y_pred_optimal = (best_y_prob >= best_threshold).astype(int)

print('=== Perbandingan: Threshold 0.5 vs Threshold Optimal ===\n')
print('Threshold 0.5 (default):')
y_pred_default = (best_y_prob >= 0.5).astype(int)
print(f'  Precision: {precision_score(y_test, y_pred_default):.4f}')
print(f'  Recall   : {recall_score(y_test, y_pred_default):.4f}')
print(f'  F1-Score : {f1_score(y_test, y_pred_default):.4f}')

print(f'\nThreshold {best_threshold:.4f} (optimal):')
print(f'  Precision: {precision_score(y_test, y_pred_optimal):.4f}')
print(f'  Recall   : {recall_score(y_test, y_pred_optimal):.4f}')
print(f'  F1-Score : {f1_score(y_test, y_pred_optimal):.4f}')

## 8. Confusion Matrix: Sebelum vs Setelah Optimasi Threshold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_default = confusion_matrix(y_test, y_pred_default)
sns.heatmap(cm_default, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Tidak Churn', 'Churn'], yticklabels=['Tidak Churn', 'Churn'])
axes[0].set_title('Threshold 0.5 (Default)', fontweight='bold')
axes[0].set_xlabel('Prediksi'); axes[0].set_ylabel('Aktual')

cm_optimal = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm_optimal, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Tidak Churn', 'Churn'], yticklabels=['Tidak Churn', 'Churn'])
axes[1].set_title(f'Threshold {best_threshold:.3f} (Optimal)', fontweight='bold')
axes[1].set_xlabel('Prediksi'); axes[1].set_ylabel('Aktual')

plt.suptitle('Dampak Optimasi Threshold pada Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Feature Importance — Model Terbaik

In [ ]:
final_model = best_xgb if best_model_name == 'XGBoost (Tuned)' else best_rf
importances = pd.Series(final_model.feature_importances_, index=X.columns)
top15 = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 7))
colors_fi = sns.color_palette('RdYlGn_r', 15)
bars = plt.barh(top15.index[::-1], top15.values[::-1], color=colors_fi[::-1])
for bar, val in zip(bars, top15.values[::-1]):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9)
plt.xlabel('Feature Importance Score')
plt.title(f'Top 15 Feature Importance — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Ringkasan Week 4

### ✅ Yang Telah Dilakukan

| Tahap | Detail |
|-------|--------|
| **Tuning Random Forest** | GridSearchCV pada 4 hyperparameter, 3-fold CV |
| **XGBoost Default** | Baseline tanpa tuning |
| **Tuning XGBoost** | GridSearchCV pada 4 hyperparameter (n_estimators, max_depth, learning_rate, subsample) |
| **Optimasi Threshold** | Mencari threshold optimal berdasarkan F1-Score, bukan default 0.5 |

### 📊 Kesimpulan Sementara

- Tuning hyperparameter memberikan peningkatan performa dibanding model default
- Optimasi threshold berhasil meningkatkan **Recall** kelas Churn — penting karena biaya kehilangan pelanggan (false negative) lebih tinggi dari biaya salah menargetkan retensi (false positive)
- Model terbaik akan digunakan sebagai final model pada Week 5

### 🗓️ Rencana Week 5
- Menyimpan model final dalam format `.pkl` (serialisasi)
- Membuat fungsi prediksi untuk data pelanggan baru
- Simulasi deployment sederhana (contoh input → prediksi churn)
- Dokumentasi cara penggunaan model